### Укажите здесь свою ФИО: Тулякова Софья Михайловна 
:<br>
:<br>
:<br>
:<br>
:<br>

## на 4/10

# Задание 1

Построить <code>pd.DataFreme</code> из двух столбцов.<br> 
В первом столбце уникальные коды IATA из базы <code>https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat</code> <br>
и <code>https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat</code>, <br> 
во втором пометка, для каждого уникального кода - "есть в двух базах", "только в airports", "только в routes"<br> 
Примечание 1. Подобное слияние баз (конечно подготовленных) можно выполнить одной строчкой кода.<br> 
Примечание 2. Помним, что <code>\N</code> нужно менять на <code>np.nan</code>.<br> 

Баллы:
1. загрузка файлов
2. слияние баз (за мининимальный и правильный код)


In [24]:
import numpy as np
import pandas as pd
from IPython.display import display


names_air=["Airport ID","Name","City","Country","IATA","ICAO","Latitude", 
                                           "Longitude", "Altitude", "Timezone", "DST", 
                                           "Tz database","Type","Source"]

routes_names = ["Airline", "Airline ID", "Source airport", "Source airport ID", 
                "Destination airport", "Destination airport ID", 
                "Codeshare", "Stops", "Equipment"]

airports_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
routes_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat"

airports = pd.read_csv(airports_url, header=None, names=names_air)

airports['IATA'] = airports['IATA'].replace('\\N', np.nan)

routes = pd.read_csv(routes_url, header=None, names=routes_names)

routes['Source airport'] = routes['Source airport'].replace('\\N', np.nan)
routes['Destination airport'] = routes['Destination airport'].replace('\\N', np.nan)

airports_iata = set(airports['IATA'].dropna().unique())

routes_source = set(routes['Source airport'].dropna().unique())
routes_dest = set(routes['Destination airport'].dropna().unique())
routes_iata = routes_source.union(routes_dest)

# уникальные IATA из обоих источ
all_iata = airports_iata.union(routes_iata)


result = pd.DataFrame({'IATA': list(all_iata)})


result['Status'] = np.where(
    result['IATA'].isin(airports_iata) & result['IATA'].isin(routes_iata),
    'есть в двух базах',
    np.where(
        result['IATA'].isin(airports_iata),
        'только в airports',
        'только в routes'
    )
)
display(result)

,IATA,Status
0,MHX,только в airports
1,CKC,только в airports
2,PPR,только в airports
3,GLD,только в airports
4,NSN,есть в двух базах
...,...,...
6230,TAO,есть в двух базах
6231,CEU,только в airports
6232,KSL,только в airports
6233,YYW,только в airports


# Задание 2 
Необходимо задать собственный тикер ценной бумаги из SP500. Остальное остается без изменений<br>
Баллы:
1. Загрузите дневные данные и удалите колонку <code>Adj Close</code><br>
2. Перестройте (агрегируйте) данные с дневных на месячные. Результат сохраните в аналогичных колонках. 

In [ ]:
import numpy as np
import pandas as pd
# import pandas_datareader.data as web
from IPython.display import display
import datetime
import requests_cache
import yfinance as yf


start = datetime.datetime(2020, 1, 1)
end = datetime.datetime(2020, 12, 31)

symbol = 'AMZN'  

expire_after = datetime.timedelta(days=3)
session = requests_cache.CachedSession(cache_name='cache', backend='sqlite', expire_after=expire_after)

with session.cache_disabled():
    data_exchange = yf.download(symbol, start=start, end=end, progress=False)

# 1 удаляем  'Adj Close'
data_exchange = data_exchange.drop(columns=['Adj Close'], errors='ignore')

# 2  дневные данные в месячные
monthly_data = data_exchange.resample('M').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
})


display(data_exchange)
display(monthly_data)

/var/folders/wz/7gxs93rj2710v6r1rv_dzzdw0000gn/T/ipykernel_69828/541413733.py:31: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data = data_exchange.resample('M').agg({


KeyError: "Column(s) ['Close', 'High', 'Low', 'Open', 'Volume'] do not exist"

попробовала, ничего не полуилось, но ниже все получилось!

data_exchange — это многомерный дата фрейм  тк yfinance по умолчанию формирует многоуровневые столбцы, даже если загружается всего один тикер

In [31]:
import pandas as pd
import requests
from apimoex import get_market_candles
from IPython.display import display

# параметры
ticker = "GAZP"
start = "2020-01-01"
end = "2020-12-31"

# с MOEX
with requests.Session() as session:
    data = get_market_candles(
        session,
        security=ticker,
        interval=24, 
        start=start,
        end=end
    )

df = pd.DataFrame(data)

df['begin'] = pd.to_datetime(df['begin'])
df = df.set_index('begin')

df = df.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})

monthly_data = df.resample('ME').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
})

monthly_data.index.name = 'Month'

display(monthly_data)


,Open,High,Low,Close,Volume
Month,,,,,
2020-01-31,259.18,261.90,226.57,226.70,943229040
2020-02-29,223.80,239.80,200.57,202.65,1068549530
2020-03-31,208.00,211.99,158.17,181.41,2274256090
2020-04-30,178.03,196.26,177.50,190.00,1151699700
2020-05-31,188.00,202.30,181.68,199.95,1119152560
2020-06-30,200.25,208.15,190.51,194.31,949645980
2020-07-31,195.39,202.33,179.80,182.59,841671960
2020-08-31,182.50,194.68,180.51,181.01,763622270
2020-09-30,181.40,186.29,169.60,169.97,1036828710


# на 5/10

3.1 правильное количество кодов IATA для двух баз



# на 6/10

3.2 Индекс дата/время по месяцам.


# на 7/10

4.1 правильное количество кодов IATA которые есть только в <code>airports</code>


# на 8/10

4.2 Проверяется результат в колонках <code>High</code> ...<code>Volume</code> - он должен корректно расчитан для месячных данных.


# на 9/10

5.1 правильное количество кодов IATA которые есть только в <code>routes</code>

# на 10/10


5.2 Постройте график OHLC. Внимание! свеча должна корректно отображаться на месяц, а не на последний или первый день месяца.